# Machine and operation classification

Run all cells from the repository root after installing `requirements.txt`.
The notebook extracts features once and compares Random Forest and XGBoost
on both the anomaly-style window split and the recording split.

In [ ]:
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display

from utils import data_loader_utils
from utils.classification import classification_splits, compare_classifiers

WINDOW_SIZE = 4000
SAMPLE_RATE = 2000
RANDOM_STATE = 42
TRAIN_FRACTION = 0.80
TARGETS = {"machine": "machine", "operation": "process"}
AXES = ["X", "Y", "Z"]

data_path = Path("./data").resolve()
file_list = sorted(data_loader_utils.find_all_h5s_in_dir(data_path))

if not file_list:
    raise ValueError("No recordings found. Check the data directory.")

print("Recordings found:", len(file_list))

## Check one window

At 2 kHz, 4,000 samples span two seconds. X, Y, Z, and their combined magnitude
each contribute 13 time-domain and 12 frequency-domain features shared with the
anomaly detection branch, for 100 inputs. The Welch spectrum uses a Hann
window and 1,024-sample segments. Standard deviation uses the sample estimate.
Skew and excess kurtosis are set to zero for constant signals, where they are
undefined.

In [ ]:
file_df = data_loader_utils.format_file(file_list[0])
first_window = file_df.iloc[:WINDOW_SIZE][AXES].to_numpy()
if len(first_window) != WINDOW_SIZE:
    raise ValueError("The first recording has no complete two-second window.")

first_features = data_loader_utils.extract_window_features(first_window, fs=SAMPLE_RATE)
print("Window shape:", first_window.shape)
pd.DataFrame([first_features])

## Build the feature table

Use every complete, non-overlapping two-second window from each recording.
Drop any shorter tail. Time spent reading and formatting HDF5 files is
reported separately from overall feature-table preparation.

In [ ]:
preparation_start = perf_counter()
read_seconds = 0.0
rows = []
skipped_tails = 0

for file_path in file_list:
    path = Path(file_path)
    read_start = perf_counter()
    recording = data_loader_utils.format_file(path)
    read_seconds += perf_counter() - read_start

    vibration = recording[AXES].to_numpy()
    window_count, tail_samples = divmod(len(vibration), WINDOW_SIZE)
    skipped_tails += bool(tail_samples)
    if not window_count:
        continue

    metadata = {
        "source_file": path.relative_to(data_path).as_posix(),
        "filename": path.name,
        "healthy": path.parent.name == "good",
        "machine": int(recording["machine"].iloc[0]),
        "process": int(recording["process"].iloc[0]),
    }
    for window_id in range(window_count):
        start = window_id * WINDOW_SIZE
        row = data_loader_utils.extract_window_features(
            vibration[start:start + WINDOW_SIZE], fs=SAMPLE_RATE
        )
        rows.append({**row, **metadata, "window": window_id})

if not rows:
    raise ValueError("No complete windows found. Check the data directory.")

features_df = pd.DataFrame(rows)
print("Complete windows:", len(features_df))
print("Recordings with complete windows:", features_df["source_file"].nunique())
print("Incomplete tails skipped:", skipped_tails)
print(f"HDF5 reading and formatting: {read_seconds:.1f} s")
print(f"Feature-table preparation including HDF5 reads: {perf_counter() - preparation_start:.1f} s")
features_df.head()

## Split the data

The anomaly-style split draws healthy training windows equal to 80% of all
windows. Remaining healthy windows and filename-sorted faulty windows are
divided between selection and final test. A recording can appear in several
partitions. Sorted input paths and seed 42 keep the sampled windows reproducible.

The recording split holds out 20% of healthy recordings as whole files.
It evaluates generalization to unseen healthy recordings. The two protocols
evaluate different populations because the anomaly holdouts include faulty windows.

In [ ]:
feature_columns = list(first_features)
if len(feature_columns) != 100:
    raise ValueError(f"Expected 100 vibration features, got {len(feature_columns)}.")

partitions = classification_splits(features_df, TRAIN_FRACTION, RANDOM_STATE)
for protocol, (train, holdouts) in partitions.items():
    print(f"\n{protocol}")
    training_files = set(train["source_file"])
    for split, partition in {"train": train, **holdouts}.items():
        health_counts = partition["healthy"].value_counts()
        print(
            f"{split}: {len(partition)} windows "
            f"({health_counts.get(True, 0)} healthy, {health_counts.get(False, 0)} faulty) "
            f"from {partition['source_file'].nunique()} recordings"
        )
        if split != "train":
            shared_files = training_files.intersection(partition["source_file"])
            print(f"  Recordings also in training: {len(shared_files)}")

## Train and evaluate the classifiers

Both algorithms see the same 100 vibration features and the same rows for
each target and protocol. The original unweighted Random Forest uses 300 trees,
up to 1,024 bootstrap samples per tree, and 70% of the features per split.
XGBoost uses 300 boosting rounds with histogram-based splits and inverse-frequency
weights computed from the training windows for each target.

Accuracy uses every evaluation window. Macro F1 averages the F1 scores of
classes present in the holdout, matching the original Random Forest report.
A holdout class missing from training stays in the evaluation and receives
zero F1. The class table shows training and holdout support. Fit seconds
measure the call to `fit` and repeat when one trained model scores two holdouts.

In [ ]:
metrics_by_protocol = {}
support_by_protocol = {}
for protocol, (train, holdouts) in partitions.items():
    metrics, support = compare_classifiers(train, holdouts, feature_columns, TARGETS, RANDOM_STATE)
    metrics_by_protocol[protocol] = metrics
    support_by_protocol[protocol] = support

results = pd.concat(metrics_by_protocol, names=["protocol"]).droplevel(1).reset_index()
class_support = (
    pd.concat(support_by_protocol, names=["protocol"]).droplevel(1).reset_index()
)
display(results.style.format(
    {"accuracy": "{:.2%}", "macro_f1": "{:.2%}", "fit_seconds": "{:.3f}"},
    na_rep="N/A",
))
display(class_support.style.format({"f1": "{:.2%}"}, na_rep="N/A"))